In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import datetime
from jugaad_data.nse import stock_df
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from datetime import date
import yfinance as yf

In [2]:
nifty_stocks = pd.read_csv("nifty_500_with_listing_date.csv")
nifty_stocks.head()

,Company Name,Industry,Symbol,ISIN Code,DATE OF LISTING
0,360 ONE WAM Ltd.,Financial Services,360ONE,INE466L01038,2019-09-19
1,ABB India Ltd.,Capital Goods,ABB,INE117A01022,1995-02-08
2,ACC Ltd.,Construction Materials,ACC,INE012A01025,1996-11-20
3,ACME Solar Holdings Ltd.,Power,ACMESOLAR,INE622W01025,2024-11-13
4,AIA Engineering Ltd.,Capital Goods,AIAENG,INE212H01026,2005-12-14


In [3]:
nifty_stocks['DATE OF LISTING'] = pd.to_datetime(nifty_stocks['DATE OF LISTING'], format='%Y-%m-%d').dt.date
nifty_stocks["YF_Ticker"] = nifty_stocks["Symbol"].astype(str).str.strip() + ".NS"
nifty_stocks[:1]

,Company Name,Industry,Symbol,ISIN Code,DATE OF LISTING,YF_Ticker
0,360 ONE WAM Ltd.,Financial Services,360ONE,INE466L01038,2019-09-19,360ONE.NS


In [12]:
today = datetime.date.today()
# print((today))

stocks_data = pd.DataFrame()
failed_stocks = []

for _, row in nifty_stocks.iterrows():
    stock_symbol = row['YF_Ticker']
    start_date = row['DATE OF LISTING']
    print(f"Pulling data for {stock_symbol} from {start_date} to {today}")

    try:
        data = yf.download(
            tickers=stock_symbol,
            start=start_date,
            end=today,
            interval="1d",
            group_by="ticker",
            threads=True,
            auto_adjust=True,
            progress=False
        )

        if data is None or data.empty:
            print(f"No data available for {stock_symbol}")
            continue

        if isinstance(data.columns, pd.MultiIndex):
            clean_df = data.stack(level=0, future_stack=True).reset_index()
        else:
            clean_df = data.reset_index().copy()

        clean_df.columns.name = None

        if "Ticker" in clean_df.columns:
            clean_df["Ticker"] = clean_df["Ticker"].astype(str).str.replace(".NS", "", regex=False)
        elif "Symbol" in clean_df.columns:
            clean_df["Ticker"] = clean_df["Symbol"].astype(str).str.replace(".NS", "", regex=False)

        if "Date" not in clean_df.columns and "index" in clean_df.columns:
            clean_df = clean_df.rename(columns={"index": "Date"})

        clean_df = clean_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
        stocks_data = pd.concat([stocks_data, clean_df], ignore_index=True)

    except Exception as exc:
        failed_stocks.append({"Ticker": stock_symbol, "Error": str(exc)})
        print(f"Failed to pull {stock_symbol}: {exc}")

stocks_data.to_csv("nifty_500_stocks_data.csv", index=False)

if failed_stocks:
    print(f"Completed with {len(failed_stocks)} failed ticker(s).")
    print(failed_stocks)
else:
    print("Completed successfully for all tickers.")


Pulling data for 360ONE.NS from 2019-09-19 to 2026-11-10
Pulling data for ABB.NS from 1995-02-08 to 2026-11-10
Pulling data for ACC.NS from 1996-11-20 to 2026-11-10
Pulling data for ACMESOLAR.NS from 2024-11-13 to 2026-11-10
Pulling data for AIAENG.NS from 2005-12-14 to 2026-11-10
Pulling data for APLAPOLLO.NS from 2011-12-14 to 2026-11-10
Pulling data for AUBANK.NS from 2017-07-10 to 2026-11-10
Pulling data for AWL.NS from 2022-02-08 to 2026-11-10
Pulling data for AADHARHFC.NS from 2024-05-15 to 2026-11-10
Pulling data for AARTIIND.NS from 1995-02-08 to 2026-11-10
Pulling data for AAVAS.NS from 2018-10-08 to 2026-11-10
Pulling data for ABBOTINDIA.NS from 2010-01-08 to 2026-11-10
Pulling data for ACE.NS from 2006-09-26 to 2026-11-10
Pulling data for ACUTAAS.NS from 2021-09-14 to 2026-11-10
Pulling data for ADANIENSOL.NS from 2015-07-31 to 2026-11-10
Pulling data for ADANIENT.NS from 1997-06-04 to 2026-11-10
Pulling data for ADANIGREEN.NS from 2018-06-18 to 2026-11-10
Pulling data for A

$BAGMANE.NS: possibly delisted; no timezone found

1 Failed download:
['BAGMANE.NS']: possibly delisted; no timezone found


Pulling data for BAGMANE.NS from NaT to 2026-11-10
No data available for BAGMANE.NS
Pulling data for BAJAJ-AUTO.NS from 2008-05-26 to 2026-11-10
Pulling data for BAJFINANCE.NS from 2003-04-01 to 2026-11-10
Pulling data for BAJAJFINSV.NS from 2008-05-26 to 2026-11-10
Pulling data for BAJAJHLDNG.NS from 1995-03-15 to 2026-11-10
Pulling data for BAJAJHFL.NS from 2024-09-16 to 2026-11-10
Pulling data for BALKRISIND.NS from 2006-01-27 to 2026-11-10
Pulling data for BALRAMCHIN.NS from 1995-02-08 to 2026-11-10
Pulling data for BANDHANBNK.NS from 2018-03-27 to 2026-11-10
Pulling data for BANKBARODA.NS from 1997-02-19 to 2026-11-10
Pulling data for BANKINDIA.NS from 1997-04-30 to 2026-11-10
Pulling data for MAHABANK.NS from 2004-04-12 to 2026-11-10
Pulling data for BATAINDIA.NS from 2003-06-18 to 2026-11-10
Pulling data for BELRISE.NS from 2025-05-28 to 2026-11-10
Pulling data for BERGEPAINT.NS from 1999-11-24 to 2026-11-10
Pulling data for BHARATCOAL.NS from 2026-01-19 to 2026-11-10
Pulling da


1 Failed download:
['BIRET.NS']: ValueError('NaTType does not support timestamp')


No data available for BIRET.NS
Pulling data for CCL.NS from 2004-06-07 to 2026-11-10
Pulling data for CESC.NS from 1995-02-08 to 2026-11-10
Pulling data for CGPOWER.NS from 1995-03-01 to 2026-11-10
Pulling data for CIEINDIA.NS from 2007-08-30 to 2026-11-10
Pulling data for CRISIL.NS from 1995-06-28 to 2026-11-10
Pulling data for CANFINHOME.NS from 1995-05-10 to 2026-11-10
Pulling data for CANBK.NS from 2002-12-23 to 2026-11-10
Pulling data for CANHLIFE.NS from 2025-10-17 to 2026-11-10
Pulling data for CAPLIPOINT.NS from 2014-06-23 to 2026-11-10
Pulling data for CGCL.NS from 2010-10-29 to 2026-11-10
Pulling data for CARBORUNIV.NS from 1996-04-03 to 2026-11-10
Pulling data for CARTRADE.NS from 2021-08-20 to 2026-11-10
Pulling data for CASTROLIND.NS from 2014-03-14 to 2026-11-10
Pulling data for CEATLTD.NS from 2008-02-01 to 2026-11-10
Pulling data for CEMPRO.NS from 1999-02-10 to 2026-11-10
Pulling data for CENTRALBK.NS from 2007-08-21 to 2026-11-10
Pulling data for CDSL.NS from 2017-06-

$DUMMYHEG.NS: possibly delisted; no timezone found

1 Failed download:
['DUMMYHEG.NS']: possibly delisted; no timezone found


No data available for DUMMYHEG.NS
Pulling data for EIDPARRY.NS from 1995-05-10 to 2026-11-10
Pulling data for EIHOTEL.NS from 1995-02-08 to 2026-11-10
Pulling data for EMBASSY.NS from NaT to 2026-11-10



1 Failed download:
['EMBASSY.NS']: ValueError('NaTType does not support timestamp')


No data available for EMBASSY.NS
Pulling data for EICHERMOT.NS from 2004-09-07 to 2026-11-10
Pulling data for ELECON.NS from 2006-09-29 to 2026-11-10
Pulling data for ELGIEQUIP.NS from 1995-11-22 to 2026-11-10
Pulling data for EMAMILTD.NS from 2006-08-03 to 2026-11-10
Pulling data for EMCURE.NS from 2024-07-10 to 2026-11-10
Pulling data for EMMVEE.NS from 2025-11-18 to 2026-11-10
Pulling data for ENDURANCE.NS from 2016-10-19 to 2026-11-10
Pulling data for ENGINERSIN.NS from 1997-08-20 to 2026-11-10
Pulling data for ESCORTS.NS from 1995-03-01 to 2026-11-10
Pulling data for ETERNAL.NS from 2021-07-23 to 2026-11-10
Pulling data for EXIDEIND.NS from 2003-06-17 to 2026-11-10
Pulling data for NYKAA.NS from 2021-11-10 to 2026-11-10
Pulling data for FEDERALBNK.NS from 1995-02-08 to 2026-11-10
Pulling data for FACT.NS from 1995-04-12 to 2026-11-10
Pulling data for FINCABLES.NS from 1995-02-15 to 2026-11-10
Pulling data for FSL.NS from 2007-02-22 to 2026-11-10
Pulling data for FIVESTAR.NS from 2

In [13]:
stocks_data.to_parquet("nifty_500_stocks_data.parquet", index=False)